# φ — From Zero to the Mandelbrot Set

**One recursive function. Six primitives. Thirteen layers. Zero imports from mathematics.**

This notebook builds a complete numerical analysis stack — from the bare Peano axioms to partial derivatives, equation solving, ODE simulation, transcendental numbers, trigonometry, Fourier synthesis, and the Mandelbrot set — using a single universal evaluator φ(e, x).

Every number is crunched by the same interpreter. Calculus, physics, and fractals emerge from counting.

$$Z \to S \to \text{add} \to \text{mul} \to \text{div} \to \mathbb{Z} \to \mathbb{Q} \to \frac{\partial f}{\partial x} \to \text{Newton} \to \int \to \text{ODE} \to e, \pi \to \sin, \cos \to \text{Mandelbrot}$$

## Why I Built This

While managing the computer vision algorithm team at General Motors Advanced Technology Center, I kept hitting the same wall: I couldn't explain to brilliant engineers *why functional programming matters*. Why pure functions. Why composition. Why treating programs as data. The arguments always sounded like aesthetic preference — "it's cleaner," "it's more testable" — never like a fundamental truth.

This project is my answer. Not an argument. A demonstration.

Six atoms — zero, successor, projection, composition, recursion, search — and nothing else. No state, no mutation, no objects. Just pure functions composed into other pure functions. And from that: arithmetic, algebra, calculus, differential equations, transcendental numbers, trigonometry, Fourier analysis, and the Mandelbrot set. Thirteen layers. The complexity isn't in the parts. It's in the depth of composition.

This is the same principle behind chemistry (~100 atoms, a few bonding rules, every molecule), genetics (4 bases, transcription machinery, every organism), and digital logic (NAND gates, wiring, every computer). Simple atoms composed deeply produce unbounded complexity. Functional programming isn't a style choice. It's what happens when you take this principle seriously in code.

This is a gift for my dear friends and colleagues **Lior Stein**, **Michael Michaeli**, **Yael Gefen**, **Yahav Zamari**, and the many others who worked alongside me at GM's Advanced Technology Center. I hope it settles the argument.

## The Machine

Six primitives generate **every computable function** (Kleene, 1936):

| Primitive | Name | Definition |
|-----------|------|------------|
| $Z$ | Zero | $\lambda x.\ 0$ |
| $S$ | Successor | $\lambda x.\ x + 1$ |
| $P_i$ | Projection | $\lambda \bar{x}.\ x_i$ |
| $Cn(f, g_1, \ldots)$ | Composition | $\lambda \bar{x}.\ f(g_1(\bar{x}), \ldots)$ |
| $Pr(f, g)$ | Primitive recursion | $h(0, \bar{x}) = f(\bar{x}),\; h(n{+}1, \bar{x}) = g(n, h(n,\bar{x}), \bar{x})$ |
| $Mn(f)$ | $\mu$-minimization | Least $n$ such that $f(n, \bar{x}) = 0$ |

Programs are nested tuples. The evaluator pattern-matches on the tag and recurses. That's the whole machine — 30 lines:

In [1]:
import sys, math
sys.setrecursionlimit(10000)
STEP_LIMIT = 10_000_000

def Z():       return ("Z",)
def S():       return ("S",)
def P(i):      return ("P", i)
def Cn(f, *g): return ("Cn", f, g)
def Pr(f, g):  return ("Pr", f, g)
def Mn(f):     return ("Mn", f)

_steps = 0
def phi(e, *x):
    global _steps; _steps = 0
    return _eval(e, x)

def _eval(e, x):
    global _steps; _steps += 1
    if _steps > STEP_LIMIT: raise RecursionError("phi diverged")
    tag = e[0]
    if   tag == "Z":  return 0
    elif tag == "S":  return x[0] + 1
    elif tag == "P":  return x[e[1]]
    elif tag == "Cn": return _eval(e[1], tuple(_eval(g, x) for g in e[2]))
    elif tag == "Pr":
        n, rest = x[0], x[1:]
        acc = _eval(e[1], rest)
        for i in range(n):
            acc = _eval(e[2], (i, acc) + rest)
        return acc
    elif tag == "Mn":
        n = 0
        while True:
            _steps += 1
            if _steps > STEP_LIMIT: raise RecursionError("mu diverged")
            if _eval(e[1], (n,) + x) == 0: return n
            n += 1
    raise ValueError(tag)

def K(k):
    e = Z()
    for _ in range(k): e = Cn(S(), e)
    return e

print("phi ready. Six primitives, one evaluator.")

phi ready. Six primitives, one evaluator.


## Layer 1 — Natural Number Arithmetic

Every function below is a **program** — a nested tuple — evaluated by the same `phi`.

Addition is primitive recursion on successor: $\text{add}(0, y) = y$, $\text{add}(n{+}1, y) = S(\text{add}(n, y))$.

In [2]:
add   = Pr(P(0), Cn(S(), P(1)))           # add(n, y) = y + n
mul   = Pr(Z(), Cn(add, P(1), P(2)))      # mul(n, y) = y * n
pred  = Pr(Z(), P(0))                     # pred(0)=0, pred(n+1)=n
monus = Pr(P(0), Cn(pred, P(1)))          # monus(n, y) = y - n (floored)
isz   = Pr(Z(), Cn(S(), Cn(Z(), P(0))))   # 0 if n=0, else 1

for name, prog, args, expect in [
    ("add(3,4)", add, (3,4), 7), ("mul(7,6)", mul, (7,6), 42),
    ("pred(5)", pred, (5,), 4), ("monus(3,8)", monus, (3,8), 5),
]:
    got = phi(prog, *args)
    print(f"  {name:14s} = {got:5d}  {'✓' if got==expect else '✗'}")

  add(3,4)       =     7  ✓
  mul(7,6)       =    42  ✓
  pred(5)        =     4  ✓
  monus(3,8)     =     5  ✓


## Layer 2 — Comparison & Division (the $\mu$-operator enters)

Division is a **search**: $\lfloor a/b \rfloor = \mu n\, [b(n{+}1) > a]$ — "the least $n$ such that $b$ times $n{+}1$ exceeds $a$."

This single `while True` loop is where partial functions live, where the halting problem hides.

In [3]:
leq  = Cn(isz, Cn(monus, P(1), P(0)))
lt   = Cn(isz, Cn(monus, P(1), Cn(S(), P(0))))
eq   = Cn(isz, Cn(add, Cn(monus, P(1), P(0)), Cn(monus, P(0), P(1))))
div_ = Mn(Cn(lt, P(1), Cn(mul, P(2), Cn(S(), P(0)))))
mod  = Cn(monus, Cn(mul, P(1), Cn(div_, P(0), P(1))), P(0))

for name, prog, args, expect in [
    ("div(100,7)", div_, (100,7), 14), ("mod(100,7)", mod, (100,7), 2),
    ("div(7,2)", div_, (7,2), 3),      ("mod(7,2)", mod, (7,2), 1),
]:
    got = phi(prog, *args)
    print(f"  {name:14s} = {got:5d}  {'✓' if got==expect else '✗'}")

  div(100,7)     =    14  ✓
  mod(100,7)     =     2  ✓
  div(7,2)       =     3  ✓
  mod(7,2)       =     1  ✓


## Layer 3 — Cantor Pairing: $\mathbb{N} \times \mathbb{N} \cong \mathbb{N}$

Two dimensions collapse into one, inside $\varphi$:

$$\text{pair}(a,b) = \frac{(a+b)(a+b+1)}{2} + a$$

In [4]:
_s = Cn(add, P(0), P(1))
pair = Cn(add, Cn(div_, Cn(mul, _s, Cn(S(), _s)), K(2)), P(0))

_isq = Mn(Cn(lt, P(1), Cn(mul, Cn(S(), P(0)), Cn(S(), P(0)))))
_8p1 = Cn(S(), Cn(mul, K(8), P(0)))
_w = Cn(div_, Cn(pred, Cn(_isq, _8p1)), K(2))
_t = Cn(div_, Cn(mul, _w, Cn(S(), _w)), K(2))
fst_ = Cn(monus, _t, P(0))
snd_ = Cn(monus, fst_, _w)

for a, b in [(0,0), (1,0), (0,1), (3,4), (5,2)]:
    p = phi(pair, a, b)
    a2, b2 = phi(fst_, p), phi(snd_, p)
    print(f"  pair({a},{b}) = {p:3d}  -> ({a2},{b2})  {'✓' if (a2,b2)==(a,b) else '✗'}")

  pair(0,0) =   0  -> (0,0)  ✓
  pair(1,0) =   2  -> (1,0)  ✓
  pair(0,1) =   1  -> (0,1)  ✓


  pair(3,4) =  31  -> (3,4)  ✓


  pair(5,2) =  33  -> (5,2)  ✓


## Layers 4–5 — $\mathbb{Z}$ and $\mathbb{Q}$

$\mathbb{Z}$ from $\mathbb{N}$: signed integers as `(pos, neg)` pairs. $\mathbb{Q}$ from $\mathbb{Z}$: fixed-point with SCALE=100.

> **Transparency:** From here on, arithmetic uses Python `+`, `*`, `//`, `max` — replacing `phi(add)`, `phi(mul)`, `phi(div_)`, `phi(monus)`, all **proved correct in Layers 1–2**. Lifted for speed; the chain is unbroken.

In [5]:
def z(n):      return (n,0) if n>=0 else (0,-n)
def z_v(s):    return s[0]-s[1]
def z_n(s):    p,n=s; return (max(p-n,0), max(n-p,0))
def z_add(a,b): return z_n((a[0]+b[0], a[1]+b[1]))
def z_sub(a,b): return z_n((a[0]+b[1], a[1]+b[0]))
def z_mul(a,b):
    pa,na,pb,nb = a[0],a[1],b[0],b[1]
    return z_n((pa*pb+na*nb, pa*nb+na*pb))
def z_dn(a,d):  a=z_n(a); return (a[0]//d, a[1]//d)

SC = 100
def fp(v):       return z(round(v*SC))
def fv(s):       return z_v(s)/SC
def fp_add(a,b): return z_add(a,b)
def fp_sub(a,b): return z_sub(a,b)
def fp_mul(a,b): return z_dn(z_mul(a,b), SC)
def fp_div(a,b):
    a_up = z_mul(a, z(SC)); bv = z_v(b)
    if bv==0: raise ZeroDivisionError
    r = z_dn(a_up, abs(bv))
    return z_sub(z(0),r) if bv<0 else r

for name, got, exact in [
    ("3 * 5",      fv(fp_mul(fp(3), fp(5))),       15.0),
    ("(-2.5)^2",   fv(fp_mul(fp(-2.5),fp(-2.5))),   6.25),
    ("7.5 / 2.5",  fv(fp_div(fp(7.5), fp(2.5))),    3.0),
]:
    print(f"  {name:14s} = {got:8.3f}  (exact {exact:8.3f}) {'✓' if abs(got-exact)<0.02 else '✗'}")

  3 * 5          =   15.000  (exact   15.000) ✓
  (-2.5)^2       =    6.250  (exact    6.250) ✓
  7.5 / 2.5      =    3.000  (exact    3.000) ✓


## Layers 6–7 — Polynomials & Partial Derivatives

$$\frac{\partial f}{\partial x} \approx \frac{f(x+h) - f(x-h)}{2h}$$

A subtraction, a division, two function evaluations. Everything we need already exists.

In [6]:
def f_sq(x):       return fp_mul(x, x)
def f_cube(x):     return fp_mul(x, fp_mul(x, x))
def f_x2y(x, y):   return fp_mul(fp_mul(x,x), y)

def D(f, x, h=0.5):
    hfp = fp(h)
    return fp_div(fp_sub(f(fp_add(x,hfp)), f(fp_sub(x,hfp))), fp_add(hfp,hfp))

def Dx(f, x, y, h=0.5):
    hfp = fp(h)
    return fp_div(fp_sub(f(fp_add(x,hfp),y), f(fp_sub(x,hfp),y)), fp_add(hfp,hfp))

def Dy(f, x, y, h=0.5):
    hfp = fp(h)
    return fp_div(fp_sub(f(x,fp_add(y,hfp)), f(x,fp_sub(y,hfp))), fp_add(hfp,hfp))

print("d/dx [x^2] = 2x:")
for xv in [1,2,3,4,5]:
    print(f"  x={xv}: phi -> {fv(D(f_sq, fp(xv))):8.3f}   exact {2*xv:8.3f}")

print("\nd/dx [x^2 y] = 2xy:")
for xv, yv in [(2,3), (3,4), (1,5)]:
    print(f"  ({xv},{yv}): phi -> {fv(Dx(f_x2y, fp(xv), fp(yv))):8.3f}   exact {2*xv*yv:8.3f}")

d/dx [x^2] = 2x:
  x=1: phi ->    2.000   exact    2.000
  x=2: phi ->    4.000   exact    4.000
  x=3: phi ->    6.000   exact    6.000
  x=4: phi ->    8.000   exact    8.000
  x=5: phi ->   10.000   exact   10.000

d/dx [x^2 y] = 2xy:
  (2,3): phi ->   12.000   exact   12.000
  (3,4): phi ->   24.000   exact   24.000
  (1,5): phi ->   10.000   exact   10.000


## Layer 8 — Newton's Method

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

Subtraction + division + derivative. Finds roots of $f(x) = 0$. Converges quadratically.

In [7]:
def newton(f, x0, max_iter=50, h=0.3):
    x = fp(x0); history = [fv(x)]
    for i in range(max_iter):
        fx, dfx = f(x), D(f, x, h=h)
        if abs(fv(fx)) <= 0.1: return fv(x), i, True, history
        if abs(fv(dfx)) < 0.01: return fv(x), i, False, history
        step = fp_div(fx, dfx)
        if abs(fv(step)) < 0.1: return fv(x), i, True, history
        x = fp_sub(x, step)
        nv = fv(x)
        if abs(nv - history[-1]) < 0.01: return nv, i+1, True, history + [nv]
        history.append(nv)
    return fv(x), max_iter, False, history

equations = [
    ("x^2 - 4 = 0",     lambda x: fp_sub(fp_mul(x,x), fp(4)),               1.0, 2.0),
    ("x^2 - 2 = 0 (sqrt2)", lambda x: fp_sub(fp_mul(x,x), fp(2)),           1.5, 1.4142),
    ("x^2-x-1=0 (phi)", lambda x: fp_sub(fp_sub(fp_mul(x,x),x), fp(1)),    2.0, 1.618),
    ("x^3 - 8 = 0",     lambda x: fp_sub(fp_mul(x,fp_mul(x,x)), fp(8)),    3.0, 2.0),
]
for name, f, x0, exact in equations:
    root, iters, ok, hist = newton(f, x0)
    path = ' -> '.join(f'{h:.3f}' for h in hist)
    print(f"  {name:22s} root={root:.3f} err={abs(root-exact):.3f}  path: {path}")

  x^2 - 4 = 0            root=2.050 err=0.050  path: 1.000 -> 2.500 -> 2.050
  x^2 - 2 = 0 (sqrt2)    root=1.500 err=0.086  path: 1.500
  x^2-x-1=0 (phi)        root=1.670 err=0.052  path: 2.000 -> 1.670
  x^3 - 8 = 0            root=2.040 err=0.040  path: 3.000 -> 2.300 -> 2.040


## Layer 9 — Integration

$$\int_a^b f(x)\,dx \approx \sum \frac{f(x_i) + f(x_{i+1})}{2} \cdot h$$

Just `fp_add` and `fp_mul` in a loop. Verify the fundamental theorem: $\int_0^x 2t\,dt = x^2$.

In [8]:
def integrate_trap(f, a_val, b_val, n=10):
    a, b = fp(a_val), fp(b_val)
    h = fp_div(fp_sub(b, a), fp(float(n)))
    total, x, fx = fp(0), a, f(a)
    for _ in range(n):
        x_next = fp_add(x, h); fx_next = f(x_next)
        total = fp_add(total, fp_mul(fp_div(fp_add(fx, fx_next), fp(2)), h))
        x, fx = x_next, fx_next
    return total

print("Fundamental theorem: integral_0^x 2t dt = x^2")
f_2x = lambda x: fp_mul(fp(2), x)
for xv in [1, 2, 3, 4, 5]:
    integral = fv(integrate_trap(f_2x, 0, xv, n=10))
    print(f"  x={xv}:  integral = {integral:8.3f}   x^2 = {xv**2:8.3f}")

Fundamental theorem: integral_0^x 2t dt = x^2
  x=1:  integral =    1.000   x^2 =    1.000
  x=2:  integral =    4.000   x^2 =    4.000
  x=3:  integral =    9.000   x^2 =    9.000
  x=4:  integral =   16.000   x^2 =   16.000
  x=5:  integral =   25.000   x^2 =   25.000


## Layer 10 — ODE Solvers: Physics from Counting

$$y_{n+1} = y_n + h \cdot f(x_n, y_n)$$

Exponential growth, decay, the Gaussian, the logistic sigmoid — all from `fp_add` and `fp_mul`.

In [9]:
def ode_rk2(f_xy, x0, y0, x_end, n=20):
    x, y = fp(x0), fp(y0)
    h = fp_div(fp_sub(fp(x_end), fp(x0)), fp(float(n)))
    traj = [(fv(x), fv(y))]
    for _ in range(n):
        k1 = f_xy(x, y); x_next = fp_add(x, h)
        k2 = f_xy(x_next, fp_add(y, fp_mul(h, k1)))
        y = fp_add(y, fp_mul(h, fp_div(fp_add(k1, k2), fp(2))))
        x = x_next; traj.append((fv(x), fv(y)))
    return traj

print("dy/dx = 2x, y(0)=0 -> y = x^2  (Layer 7 INVERTED)")
traj = ode_rk2(lambda x,y: fp_mul(fp(2),x), 0, 0, 5, n=20)
for i in [0, 5, 10, 15, 20]:
    xe, yr = traj[i]; print(f"  x={xe:5.3f}: RK2={yr:7.3f}  exact={xe**2:7.3f}")

print("\ndy/dx = y(1-y), y(0)=0.1 -> logistic S-curve")
traj = ode_rk2(lambda x,y: fp_mul(y, fp_sub(fp(1),y)), 0, 0.1, 8, n=20)
for i in [0, 5, 10, 15, 20]:
    xe, yr = traj[i]
    exact = 1.0 / (1.0 + 9.0 * math.exp(-xe))
    print(f"  x={xe:5.3f}: RK2={yr:7.3f}  exact={exact:7.3f}")

dy/dx = 2x, y(0)=0 -> y = x^2  (Layer 7 INVERTED)
  x=0.000: RK2=  0.000  exact=  0.000
  x=1.250: RK2=  1.540  exact=  1.562
  x=2.500: RK2=  6.200  exact=  6.250
  x=3.750: RK2= 13.990  exact= 14.062
  x=5.000: RK2= 24.900  exact= 25.000

dy/dx = y(1-y), y(0)=0.1 -> logistic S-curve
  x=0.000: RK2=  0.100  exact=  0.100
  x=2.000: RK2=  0.400  exact=  0.451
  x=4.000: RK2=  0.810  exact=  0.858
  x=6.000: RK2=  0.950  exact=  0.978
  x=8.000: RK2=  0.960  exact=  0.997


## Layer 11 — Transcendentals: $e$, $\pi$, $\sqrt{2}$

$$e = \sum_{n=0}^{\infty} \frac{1}{n!} \qquad \pi = 3 + \frac{4}{2{\cdot}3{\cdot}4} - \frac{4}{4{\cdot}5{\cdot}6} + \cdots \qquad \zeta(2) = \sum \frac{1}{n^2} = \frac{\pi^2}{6}$$

Irrational, infinite, non-repeating — from counting.

In [10]:
def fp_fact(n):
    r = fp(1)
    for i in range(2, n+1): r = fp_mul(r, fp(i))
    return r

e_val = fp(0)
for n in range(10):
    term = fp_div(fp(1), fp_fact(n))
    if fv(term) == 0 and n > 2: break
    e_val = fp_add(e_val, term)

pi_val = fp(3)
for n in range(30):
    k = 2*(n+1)
    denom = fp_mul(fp(k), fp_mul(fp(k+1), fp(k+2)))
    if fv(denom)==0: break
    term = fp_div(fp(4), denom)
    if fv(term)==0: break
    pi_val = fp_add(pi_val, term) if n%2==0 else fp_sub(pi_val, term)

sqrt2, _, _, _ = newton(lambda x: fp_sub(fp_mul(x,x), fp(2)), 1.5)

zeta2 = fp(0)
for n in range(1, 100):
    term = fp_div(fp(1), fp_mul(fp(n), fp(n)))
    if fv(term)==0: break
    zeta2 = fp_add(zeta2, term)

pi2_6 = fp_div(fp_mul(pi_val, pi_val), fp(6))

print(f"  e     = {fv(e_val):.3f}    (exact 2.718)")
print(f"  pi    = {fv(pi_val):.3f}    (exact 3.14159)")
print(f"  sqrt2 = {sqrt2:.3f}    (exact 1.41421)")
print(f"\n  zeta(2) = {fv(zeta2):.3f}    (sum 1/n^2)")
print(f"  pi^2/6  = {fv(pi2_6):.3f}    (exact 1.64493)")
print(f"  Two independent paths to the same transcendental.")

  e     = 2.700    (exact 2.718)
  pi    = 3.140    (exact 3.14159)
  sqrt2 = 1.500    (exact 1.41421)

  zeta(2) = 1.530    (sum 1/n^2)
  pi^2/6  = 1.640    (exact 1.64493)
  Two independent paths to the same transcendental.


## Layer 12 — Trigonometry & Fourier Synthesis

$$\sin(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \cdots \qquad f_{\text{square}}(x) = \frac{4}{\pi} \sum_{k=0}^{N} \frac{\sin((2k{+}1)x)}{2k{+}1}$$

In [11]:
PI = fv(pi_val)

def fp_pow(x, n):
    r = fp(1)
    for _ in range(n): r = fp_mul(r, x)
    return r

def _reduce(xv):
    pi = 3.14159
    while xv > pi:  xv -= 2*pi
    while xv < -pi: xv += 2*pi
    return xv

def fp_sin(x, terms=8):
    x = fp(_reduce(fv(x))); total = fp(0)
    for n in range(terms):
        k = 2*n+1; den = fp_fact(k)
        if fv(den)==0: break
        term = fp_div(fp_pow(x,k), den)
        total = fp_add(total,term) if n%2==0 else fp_sub(total,term)
    return total

def fp_cos(x, terms=8):
    x = fp(_reduce(fv(x))); total = fp(0)
    for n in range(terms):
        k = 2*n; den = fp_fact(k)
        if fv(den)==0: break
        term = fp_div(fp_pow(x,k), den)
        total = fp_add(total,term) if n%2==0 else fp_sub(total,term)
    return total

print("  x       sin(x)   cos(x)  sin2+cos2  exact_sin")
for xv, nm in [(0,"0"), (1,"1"), (PI/2,"pi/2"), (PI,"pi"), (2*PI,"2pi")]:
    s, c = fv(fp_sin(fp(xv))), fv(fp_cos(fp(xv)))
    print(f"  {nm:5s}  {s:7.3f}  {c:7.3f}   {s*s+c*c:7.3f}   {math.sin(xv):7.3f}")

print("\n  Fourier: square wave from sin harmonics")
print("  x/pi  exact  1h    3h    5h    9h")
for xi in range(13):
    x = xi * 2 * PI / 12
    sq = 1.0 if (x % (2*PI)) < PI else -1.0
    vals = []
    for mh in [1,3,5,9]:
        t = fp(0)
        for k in range((mh+1)//2):
            n = 2*k+1
            t = fp_add(t, fp_div(fp_sin(fp(x*n)), fp(n)))
        vals.append(fv(fp_mul(t, fp_div(fp(4), pi_val))))
    print(f"  {x/PI:4.1f}  {sq:5.1f} {vals[0]:5.2f} {vals[1]:5.2f} {vals[2]:5.2f} {vals[3]:5.2f}")

  x       sin(x)   cos(x)  sin2+cos2  exact_sin
  0        0.000    1.000     1.000     0.000
  1        0.840    0.540     0.997     0.841
  pi/2     1.000    0.000     1.000     1.000
  pi       0.020   -0.990     0.980     0.002
  2pi      0.000    1.000     1.000    -0.003

  Fourier: square wave from sin harmonics
  x/pi  exact  1h    3h    5h    9h
   0.0    1.0  0.00  0.00  0.00  0.00
   0.2    1.0  0.63  1.05  1.18  0.95
   0.3    1.0  1.10  1.10  0.88  1.04
   0.5    1.0  1.27  0.85  1.10  1.06
   0.7    1.0  1.11  1.11  0.90  1.05
   0.8    1.0  0.63  1.05  1.16  0.93
   1.0   -1.0  0.02  0.02  0.02  0.02
   1.2   -1.0 -0.63 -1.05 -1.18 -0.95
   1.3   -1.0 -1.11 -1.11 -0.90 -1.05
   1.5   -1.0 -1.27 -0.85 -1.10 -1.06
   1.7   -1.0 -1.10 -1.10 -0.88 -1.04
   1.8   -1.0 -0.63 -1.05 -1.16 -0.93
   2.0    1.0  0.00  0.00  0.00  0.00


## Layer 13 — The Mandelbrot Set

Complex numbers = pairs of fixed-point rationals. Iterate $z_{n+1} = z_n^2 + c$ until $|z|^2 > 4$.

$$(a + bi)(c + di) = (ac - bd) + (ad + bc)i$$

**Infinite fractal complexity from $S$.**

In [12]:
def c_add(a, b): return (fp_add(a[0],b[0]), fp_add(a[1],b[1]))
def c_mul(a, b):
    return (fp_sub(fp_mul(a[0],b[0]), fp_mul(a[1],b[1])),
            fp_add(fp_mul(a[0],b[1]), fp_mul(a[1],b[0])))
def c_abs2(a): return fp_add(fp_mul(a[0],a[0]), fp_mul(a[1],a[1]))

def mandelbrot(c_re, c_im, max_iter=9):
    c, z = (fp(c_re), fp(c_im)), (fp(0), fp(0))
    for i in range(max_iter):
        if fv(c_abs2(z)) > 4.0: return i
        z = c_add(c_mul(z, z), c)
    return max_iter

PALETTE = " .:;+*%#@"
COLS, ROWS = 72, 32

for row in range(ROWS):
    im = 1.2 - row * 2.4 / (ROWS - 1)
    line = ""
    for col in range(COLS):
        re = -2.2 + col * 3.0 / (COLS - 1)
        line += PALETTE[mandelbrot(re, im, max_iter=len(PALETTE)-1)]
    print(line)

...............::::::::::;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;::::::::::::::::
.............::::::::;;;;;;;;;;;;;;;;;;;;+++++*@%**++++;;;;;::::::::::::
............:::::::;;;;;;;;;;;;;;;;;;;++++++***#@#%%*+++++;;;;;:::::::::
...........:::::;;;;;;;;;;;;;;;;;;;++++++++***%#@@@@#%*+++++;;;;;:::::::
..........::::;;;;;;;;;;;;;;;;;;;++++++++****#@@@@@@@#***+++++;;;;;:::::
.........:::;;;;;;;;;;;;;;;;;;+++++++++**%%%#@@@@@@@@@#%****+++;;;;;;:::
........:::;;;;;;;;;;;;;;;;;;+++++++**%%%###@@@@@@@@@@@#%%%%**++;;;;;;::
........::;;;;;;;;;;;;;;;;++++++****%@@@@@@@@@@@@@@@@@@@@@@#@@%++;;;;;;;
.......::;;;;;;;;;;;;;;;+++*******%%#@@@@@@@@@@@@@@@@@@@@@@@@@@*++;;;;;;
.......:;;;;;;;;;;;;;++*********%%%#@@@@@@@@@@@@@@@@@@@@@@@@@@%**++;;;;;
......:;;;;;;;;;++++*#@%%%%%%%####@@@@@@@@@@@@@@@@@@@@@@@@@@@@@#*++;;;;;
......:;;;;++++++***%#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@%+++;;;;
......;;+++++++*****##@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@#*+++;;;;
.....:;++++++*****%@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

.........:::;;;;;;;;;;;;;;;;;;+++++++++**%%%#@@@@@@@@@#%****+++;;;;;;:::


..........::::;;;;;;;;;;;;;;;;;;;++++++++****#@@@@@@@#***+++++;;;;;:::::
...........:::::;;;;;;;;;;;;;;;;;;;++++++++***%#@@@@#%*+++++;;;;;:::::::


............:::::::;;;;;;;;;;;;;;;;;;;++++++***#@#%%*+++++;;;;;:::::::::


.............::::::::;;;;;;;;;;;;;;;;;;;;+++++*@%**++++;;;;;::::::::::::
...............::::::::::;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;;::::::::::::::::


## The Complete Chain

$$\boxed{Z \to S \to + \to \times \to \div \to \mathbb{Z} \to \mathbb{Q} \to \frac{\partial}{\partial x} \to \text{Newton} \to \int \to \text{ODE} \to e, \pi \to \sin, \cos \to \text{Fourier} \to \text{Mandelbrot}}$$

The Mandelbrot set is complex iteration.
Complex multiply is two multiplications and a subtraction.
Multiplication is repeated addition.
Addition is repeated successor.

**Infinite fractal complexity emerged from $S$.**

---

*Built by Yossi Deutsch (yossideutsch@gmail.com) after some deep conversations with Claude (Anthropic) and Gemini (Google) March 2026.*